In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
def describe(df):
    print("Shape: ", df.shape)
    print("\nFirst 5 rows:");  print(df.head())
    print("\nLast 5 rows:");   print(df.tail())
    print("\nInfo:");          df.info()
    print("\nStatistics:");    print(df.describe())
    print("\nMissing values:"); print(df.isnull().sum())

In [3]:
np.random.seed(42); n = 500
df = pd.DataFrame({"CGPA": np.round(np.random.normal(7.2,1,n),2),
                   "Internships": np.random.randint(0,4,n),
                   "Projects": np.random.randint(0,6,n),
                   "Aptitude": np.random.randint(40,100,n).astype(float),
                   "Backlogs": np.random.randint(0,3,n)})
score = (df.CGPA*1.5 + df.Internships*1.2 + df.Projects*0.8 + df.Aptitude*0.05
         - df.Backlogs*1.5 + np.random.normal(0,2,n))
df["Placed"] = (score > score.quantile(0.4)).astype(int)

In [4]:
df.loc[np.random.choice(n,15,replace=False),"CGPA"] = np.nan
df.loc[np.random.choice(n,15,replace=False),"Aptitude"] = np.nan
df.loc[np.random.choice(n,5,replace=False),"CGPA"] = 15
df = pd.concat([df, df.sample(10, random_state=1)], ignore_index=True)
describe(df)

Shape:  (510, 6)

First 5 rows:
   CGPA  Internships  Projects  Aptitude  Backlogs  Placed
0  7.70            1         1      77.0         1       0
1  7.06            3         1      54.0         0       0
2  7.85            1         2      73.0         2       1
3  8.72            2         3      74.0         0       1
4  6.97            3         3      88.0         1       1

Last 5 rows:
     CGPA  Internships  Projects  Aptitude  Backlogs  Placed
505  5.78            0         3      72.0         2       0
506  6.98            0         1      44.0         2       0
507  9.05            2         0      63.0         0       0
508  7.60            2         1      72.0         2       0
509  7.30            1         1      55.0         0       0

Info:
<class 'pandas.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CGPA         495 non-null    float64
 1   Int

In [5]:
df["CGPA"] = df["CGPA"].fillna(df["CGPA"].mean())            # missing values
df["Aptitude"] = df["Aptitude"].fillna(df["Aptitude"].mean())
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)             # duplicates
for c in ["CGPA","Aptitude"]:                                # outliers (IQR capping)
    q1,q3 = df[c].quantile([.25,.75]); iqr = q3-q1
    df[c] = df[c].clip(q1-1.5*iqr, q3+1.5*iqr)
print("Final shape:", df.shape)

X, y = df.drop("Placed",axis=1), df["Placed"]
print(pd.DataFrame(MinMaxScaler().fit_transform(X),columns=X.columns).describe().loc[["min","max"]])   # normalization (0-1)
print(pd.DataFrame(StandardScaler().fit_transform(X),columns=X.columns).describe().loc[["mean","std"]]) # standardization


Duplicates: 10
Final shape: (500, 6)
     CGPA  Internships  Projects  Aptitude  Backlogs
min   0.0          0.0       0.0       0.0       0.0
max   1.0          1.0       1.0       1.0       1.0
              CGPA   Internships      Projects      Aptitude      Backlogs
mean  4.529710e-16  2.309264e-17 -6.217249e-17  3.108624e-16  3.019807e-17
std   1.001002e+00  1.001002e+00  1.001002e+00  1.001002e+00  1.001002e+00


In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
sc = StandardScaler(); X_train = sc.fit_transform(X_train); X_test = sc.transform(X_test)  # scaling
model = LogisticRegression().fit(X_train,y_train)
print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))

Accuracy: 0.85


In [7]:
import ipywidgets as w
from IPython.display import display
def predict(cgpa,i,p,a,b):
    row = pd.DataFrame([[cgpa,i,p,a,b]], columns=X.columns)
    return "Placed" if model.predict(sc.transform(row))[0]==1 else "Not Placed"
cg = w.FloatSlider(value=7,min=0,max=10,step=0.1,description="CGPA")
it = w.IntSlider(value=1,min=0,max=5,description="Internships")
pr = w.IntSlider(value=2,min=0,max=10,description="Projects")
ap = w.IntSlider(value=70,min=0,max=100,description="Aptitude")
bk = w.IntSlider(value=0,min=0,max=5,description="Backlogs")
btn = w.Button(description="Predict", button_style="success"); out = w.Output()
def click(b):
    out.clear_output()
    with out: print("Result:", predict(cg.value,it.value,pr.value,ap.value,bk.value))
btn.on_click(click); display(cg,it,pr,ap,bk,btn,out)

FloatSlider(value=7.0, description='CGPA', max=10.0)

IntSlider(value=1, description='Internships', max=5)

IntSlider(value=2, description='Projects', max=10)

IntSlider(value=70, description='Aptitude')

IntSlider(value=0, description='Backlogs', max=5)

Button(button_style='success', description='Predict', style=ButtonStyle())

Output()

In [8]:
batch = pd.DataFrame({"CGPA":[8.5,6.2,7.8],"Internships":[2,0,1],"Projects":[3,1,4],
                      "Aptitude":[85,55,72],"Backlogs":[0,2,0]})
batch["Prediction"] = np.where(model.predict(sc.transform(batch))==1,"Placed","Not Placed")
print(batch)

   CGPA  Internships  Projects  Aptitude  Backlogs  Prediction
0   8.5            2         3        85         0      Placed
1   6.2            0         1        55         2  Not Placed
2   7.8            1         4        72         0      Placed
